# Collect MI Results from Disk & Plot MI Scaling

In [ ]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
from itertools import product
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

DATA_ROOT = Path('/home/igor/noise_scaling/data')

ALGOS = ['Geneformer', 'PCA', 'RandomProjection', 'SCVI', 'State']

EXPECTED = {
    'PBMC': {
        'sizes': [100, 215, 464, 1000, 2154, 4641, 10000, 21544, 46415, 100000],
        'qualities': [0.0012346, 0.0025982, 0.0054682, 0.0115083, 0.02422, 0.050973, 0.1072766, 0.225772, 0.4751547, 1.0],
        'signals': ['protein_counts'],
        'seeds': [42, 2303, 2701],
    },
    'larry': {
        'sizes': [100, 215, 464, 1000, 2154, 4641, 10000, 21544, 46415, 100000],
        'qualities': [0.003876, 0.0071835, 0.0133136, 0.0246748, 0.0457311, 0.0847557, 0.1570821, 0.2911284, 0.5395631, 1.0],
        'signals': ['clone'],
        'seeds': [42, 1404, 2701],
    },
    'merfish': {
        'sizes': [100, 203, 414, 843, 1716, 3494, 7113, 14480, 29475, 60000],
        'qualities': [0.027248, 0.0406617, 0.0606789, 0.0905502, 0.1351267, 0.2016475, 0.3009156, 0.4490518, 0.6701133, 1.0],
        'signals': ['ng_idx'],
        'seeds': [1404, 2303, 2701],
    },
    'shendure': {
        'sizes': [100, 359, 1291, 4641, 16681, 59948, 215443, 774263, 2782559, 10000000],
        'qualities': [0.004, 0.0073875, 0.0136438, 0.0251984, 0.0465384, 0.0859506, 0.1587401, 0.2931733, 0.5414548, 1.0],
        'signals': ['author_day'],
        'seeds': [42],
    },
}

In [ ]:
def _read_mi(path):
    try:
        with open(path) as f:
            return float(f.read().strip())
    except Exception:
        return np.nan

# Build expected paths for non-State algorithms
expected_rows = []
for ds, cfg in EXPECTED.items():
    for sz, q, algo, sig, sd in product(cfg['sizes'], cfg['qualities'], ALGOS, cfg['signals'], cfg['seeds']):
        if algo == 'State':
            continue  # State scanned separately below
        suffix = '_geneformer' if algo == 'Geneformer' else ''
        stem = f'Y_{sig}_{q}{suffix}'
        mi_path = DATA_ROOT / ds / str(sz) / str(q) / 'results' / algo / 'model' / 'MI' / str(sd) / stem / 'lmi_mutual_information.txt'
        expected_rows.append({
            'dataset': ds, 'size': sz, 'quality': q, 'algorithm': algo,
            'signal': sig, 'seed': sd, 'path': str(mi_path),
        })

df_expected = pd.DataFrame(expected_rows)
print(f'Total expected (non-State): {len(df_expected)}')

# Check existence in parallel
with ThreadPoolExecutor(max_workers=128) as pool:
    df_expected['exists'] = list(tqdm(pool.map(os.path.exists, df_expected['path']),
                                      total=len(df_expected), desc='Scanning'))

# Read MI values for found files
df_found = df_expected[df_expected['exists']].copy()
print(f'Found: {len(df_found)}, Missing: {len(df_expected) - len(df_found)}')

with ThreadPoolExecutor(max_workers=128) as pool:
    df_found['mi_value'] = list(tqdm(pool.map(_read_mi, df_found['path']),
                                     total=len(df_found), desc='Reading MI'))

# Scan State algorithm results directly from disk (may use different signals)
state_rows = []
for mi_file in DATA_ROOT.rglob('results/State/model/MI/*/Y_*/lmi_mutual_information.txt'):
    parts = mi_file.parts
    idx = parts.index('results')
    quality = parts[idx - 1]
    size = parts[idx - 2]
    dataset = parts[idx - 3]
    seed = parts[idx + 4]
    signal_raw = parts[idx + 5]  # e.g. Y_ng_idx_1.0
    sig = signal_raw.replace('Y_', '')
    sig_parts = sig.rsplit('_', 1)
    try:
        float(sig_parts[-1])
        sig = sig_parts[0]
    except ValueError:
        pass
    try:
        mi_val = float(mi_file.read_text().strip())
    except Exception:
        continue
    state_rows.append({
        'dataset': dataset, 'size': int(size), 'quality': float(quality),
        'algorithm': 'State', 'signal': sig, 'seed': int(seed), 'mi_value': mi_val,
    })

df_state = pd.DataFrame(state_rows)
print(f'State results from disk: {len(df_state)} rows')

# Combine
out_cols = ['dataset', 'size', 'quality', 'algorithm', 'signal', 'seed', 'mi_value']
df_results = pd.concat([
    df_found[out_cols],
    df_state[out_cols] if len(df_state) > 0 else pd.DataFrame(columns=out_cols),
], ignore_index=True).sort_values(out_cols[:-1]).reset_index(drop=True)
print(f'Collected {len(df_results)} MI values (NaN: {df_results["mi_value"].isna().sum()})')

In [ ]:
# Completeness for non-State algorithms
pivot = df_expected.groupby(['dataset', 'algorithm'])['exists'].agg(['sum', 'count'])
pivot.columns = ['found', 'expected']
pivot['missing'] = pivot['expected'] - pivot['found']

# Add State counts from disk scan
state_counts = df_state.groupby('dataset').size().reset_index(name='found') if len(df_state) > 0 else pd.DataFrame(columns=['dataset', 'found'])
for ds in EXPECTED:
    n = state_counts.loc[state_counts['dataset'] == ds, 'found'].values
    n = n[0] if len(n) > 0 else 0
    pivot.loc[(ds, 'State'), :] = [n, '?', '?']

pivot = pivot.sort_index()
pivot = pivot.unstack('algorithm', fill_value=0)
display(pivot)

In [ ]:
df_missing = df_expected[~df_expected['exists']][['dataset', 'size', 'quality', 'algorithm', 'signal', 'seed']].copy()
df_missing = df_missing.sort_values(df_missing.columns.tolist()).reset_index(drop=True)

if df_missing.empty:
    print('No missing runs!')
else:
    print(f'{len(df_missing)} missing runs:')
    pivot_miss = df_missing.groupby(['dataset', 'algorithm']).size().unstack(fill_value=0)
    display(pivot_miss)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# Filter out cur_idx and celltype.l3 signals
EXCLUDE_SIGNALS = {'cur_idx', 'celltype.l3'}
df_plot = df_results[~df_results['signal'].isin(EXCLUDE_SIGNALS)]

# Column order: State, Geneformer, SCVI, PCA, RandomProjection
ALGO_ORDER = ['State', 'Geneformer', 'SCVI', 'PCA', 'RandomProjection']

def plot_mi_scaling(df):
    """Plot MI vs number of cells: rows=dataset×signal, cols=algorithm."""
    all_ds_sig = sorted(set(zip(df['dataset'], df['signal'])))
    algorithms = [a for a in ALGO_ORDER if a in df['algorithm'].unique()]
    n_rows = len(all_ds_sig)
    n_cols = len(algorithms)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 3.5 * n_rows + 0.8),
                             squeeze=False, sharex=False)

    # Share y-axis within each row
    for i in range(n_rows):
        for j in range(1, n_cols):
            axes[i, j].sharey(axes[i, 0])

    all_qualities = sorted(df['quality'].unique())
    norm = mcolors.LogNorm(vmin=min(all_qualities), vmax=max(all_qualities))
    cmap = plt.cm.viridis

    for i, (ds, sig) in enumerate(all_ds_sig):
        for j, algo in enumerate(algorithms):
            ax = axes[i, j]
            sub = df[(df['dataset'] == ds) & (df['algorithm'] == algo) & (df['signal'] == sig)]

            if sub.empty:
                ax.text(0.5, 0.5, 'no data', ha='center', va='center',
                        transform=ax.transAxes, color='gray', fontsize=10)
                ax.set_xscale('log')
            else:
                for q in sorted(sub['quality'].unique()):
                    color = cmap(norm(q))
                    q_df = sub[sub['quality'] == q]
                    agg = q_df.groupby('size')['mi_value'].agg(['mean', 'std', 'count']).reset_index()
                    agg = agg.sort_values('size')
                    agg['sem'] = (agg['std'] / np.sqrt(agg['count'])).fillna(0)
                    ax.errorbar(agg['size'], agg['mean'], yerr=agg['sem'],
                                marker='o', markersize=3, linewidth=1, capsize=2,
                                color=color, alpha=0.85)
                ax.set_xscale('log')

            # X-axis label: "Number of cells" on bottom row, dataset name on all rows
            if i == n_rows - 1:
                ax.set_xlabel('Number of cells', fontsize=10)
            if j == 0:
                ax.set_ylabel(f'{ds} — MI ({sig})', fontsize=10)
            if i == 0:
                ax.set_title(algo, fontsize=12, fontweight='bold')
            ax.tick_params(labelsize=8)
            if j > 0:
                ax.tick_params(labelleft=False)

    # Colorbar: horizontal, below the plot, aligned right
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar_ax = fig.add_axes([0.55, 0.01, 0.35, 0.015])  # [left, bottom, width, height]
    cbar = fig.colorbar(sm, cax=cbar_ax, orientation='horizontal')
    cbar.set_label('Quality (downsampling ratio)', fontsize=10)
    cbar.ax.tick_params(labelsize=8)

    fig.suptitle('MI vs Number of Cells (from disk)',
                 fontsize=14, fontweight='bold')
    fig.tight_layout(rect=[0, 0.04, 1, 0.96])
    plt.show()
    return fig

fig = plot_mi_scaling(df_plot)